1. Load Clean Datsets

In [2]:
import pandas as pd
import numpy as np

# Load clean datasets
customers = pd.read_csv('../data/cleaned/customers_clean.csv')
offers = pd.read_csv('../data/cleaned/offers_clean.csv')
events = pd.read_csv('../data/cleaned/events_clean.csv')

2. Offer funnel Matching 
Event Table



 Goal: for every 'offer received' event, figure out whether it was:
    - viewed (within the offer's duration window)
    - completed AFTER being viewed (a truly "influenced" completion,
        not just a coincidental purchase that happened to clear the threshold)

        

In [3]:
# 1. One row per received offer, with its expiry time
received = (
    events.loc[events['event'].eq('offer received'), ['customer_id', 'offer_id', 'time']]  # keep only the offer received and 3 columns
    .rename(columns={'time': 'received_time'}) # later it can be clear that is it a received time or viewing time 
    .merge(offers[['offer_id', 'offer_type', 'difficulty', 'reward', 'duration']], 
           on='offer_id', how='left')   # merge with offers
)
received['window_end'] = received['received_time'] + received['duration'] * 24  # the deadline. duration is in days and time is in hours, so we multiply by 24. Customer A's first receipt: 0 + 3 × 24 = 72.
received = received.sort_values('received_time').reset_index(drop=True)
received['received_event_id'] = received.index

In [4]:
# Attach each view and completion to one receipt
def attach_to_receipt(event_name):
    ev = (events.loc[events['event'].eq(event_name), ['customer_id', 'offer_id', 'time']]
          .sort_values('time'))
    m = pd.merge_asof(     # nearest match merge eg. if A has received at o and 100 hour. Viewed at 120 hour then pick the lastest one i.e 100 hours. as it doesnot need to read two times
        ev,
        received[['received_event_id', 'customer_id', 'offer_id', 'received_time', 'window_end']],
        left_on='time', right_on='received_time',
        by=['customer_id', 'offer_id'],
        direction='backward')
    in_window = m['received_event_id'].notna() & (m['time'] <= m['window_end'])  # throws out matches that comes too late eg Customer C viewed at hour 100, and merge_asof matched it to receipt 2, but that receipt expired at hour 72. 100 > 72, so the view is dropped. That leaves C as "received only". notna() drop if view has no matching receipt
    print(...)
    return m[in_window]

views = attach_to_receipt('offer viewed')
completions = attach_to_receipt('offer completed')


Ellipsis
Ellipsis


In [5]:
# 3. First view, and completions that follow a view
first_view = views.groupby('received_event_id')['time'].min().rename('first_view_time') # one receipt can have several views, keep the earliest per receipt
first_any_completion = (completions.groupby('received_event_id')['time'].min()
                        .rename('first_completion_any'))    # earliest completion per receipt, regardless of whether it was viewed.

inf = completions.merge(first_view, on='received_event_id')  # influenced test
inf = inf[inf['time'] >= inf['first_view_time']]     # A completes at 30 and first viewed at 10, so 30 ≥ 10 and it's influenced. if greater or no view not influenced
first_influenced = (inf.groupby('received_event_id')['time'].min() 
                    .rename('influenced_completion_time'))


In [6]:
# 4. build the final table and label each row

funnel = received.set_index('received_event_id').join([first_view, first_any_completion, first_influenced]) # starts from the receipts table and adds the three time columns. Receipts with no match get NaN.

funnel['viewed'] = funnel['first_view_time'].notna()     #notna() turns "does a value exist?" into True/False. viewed = True means a first-view time exists.
funnel['completed_any'] = funnel['first_completion_any'].notna()
funnel['influenced_completion'] = funnel['influenced_completion_time'].notna()

funnel['funnel_stage'] = np.select(     #an if / elif / else. It checks the conditions in order, and the first one that's true wins.
    [funnel['influenced_completion'],
     funnel['completed_any'] & ~funnel['influenced_completion'],
     funnel['viewed']],
    ['completed after view', 'completed without view', 'viewed only'],
    default='received only')



In [7]:
print(funnel)

                                        customer_id  \
received_event_id                                     
0                  78afa995795e4d85b5d9ceeca43f5fef   
1                  e2127556f4f64592b11af22de27a7932   
2                  389bc3fa690240e798340f5a15918d5c   
3                  2eeac8d8feae4a8cad5a6af0499a211d   
4                  aa4862eba776480b8bb9c68455b8c2e1   
...                                             ...   
66496              d087c473b4d247ccb0abfef59ba12b0e   
66497              cb23b66c56f64b109d673d5e56574529   
66498              6d5f3a774f3d4714ab0c092238f3a1d7   
66499              9dc1421481194dcd9400aec7c9ae6366   
66500              e4052622e5ba45a8b96b59aba68cf068   

                                           offer_id  received_time  \
received_event_id                                                    
0                  9b98b8c7a33c4b65b9aebfe6a799e6d9              0   
1                  2906b810c7d4411798c6938adc9daaa5              0   
2   

In [8]:
# 5. Sanity check: funnel must have exactly one row per 'offer received' event (a bad merge would duplicate rows)
assert len(funnel) == events['event'].eq('offer received').sum()

In [9]:
# 6. Summary Table
summary = funnel.groupby('offer_type')[['viewed', 'completed_any', 'influenced_completion']].sum()
summary.insert(0, 'received', funnel.groupby('offer_type').size())

summary['view_rate'] = summary['viewed'] / summary['received']
summary['influenced_rate'] = summary['influenced_completion'] / summary['received']

summary.round(3)

,received,viewed,completed_any,influenced_completion,view_rate,influenced_rate
offer_type,,,,,,
bogo,26537,21865,15100,10647,0.824,0.401
discount,26664,18393,16900,11702,0.690,0.439
informational,13300,8585,0,0,0.645,0.000


In [10]:
# save offer funnel output in csv file
# received_event_id is currently the index; make it a normal column so it survives the CSV
funnel_out = funnel.reset_index()
funnel_out.to_csv('../data/cleaned/offer_funnel.csv', index=False)
print("offer_funnel:", funnel_out.shape)

offer_funnel: (66501, 16)


In [12]:
check = pd.read_csv('../data/cleaned/offer_funnel.csv')
assert len(check) == len(funnel)   # same row count after saving and reloading


In [13]:
print(check.dtypes)

received_event_id               int64
customer_id                    object
offer_id                       object
received_time                   int64
offer_type                     object
difficulty                      int64
reward                          int64
duration                        int64
window_end                      int64
first_view_time               float64
first_completion_any          float64
influenced_completion_time    float64
viewed                           bool
completed_any                    bool
influenced_completion            bool
funnel_stage                   object
dtype: object
